# CosyVoice 단독 테스트

**순서대로 셀을 실행하세요.**

- 런타임: GPU (T4 이상)
- 테스트할 음성 파일(.wav)을 Google Drive에 미리 올려두세요.

## Step 1. GPU 확인

In [ ]:
import torch
print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: GPU 없음 — 런타임 유형을 GPU로 변경하세요')

## Step 2. Google Drive 마운트

`AUDIO_PATH`를 본인 음성 파일 경로로 수정하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ★ 본인 경로로 수정
AUDIO_PATH = '/content/drive/MyDrive/test_audio.wav'
OUTPUT_DIR = '/content/drive/MyDrive/cosyvoice_test_output'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('입력 파일 존재:', os.path.exists(AUDIO_PATH))
print('출력 폴더:', OUTPUT_DIR)

## Step 3. VoiceSecure SDK GitHub pull

`test/cosyvoice-standalone` 브랜치에서 최신 코드를 가져옵니다.

In [ ]:
import os

SDK_ROOT = '/content/voicesecure-sdk'

if not os.path.exists(SDK_ROOT):
    !git clone -b test/cosyvoice-standalone https://github.com/VoiceSecureHoseo/voicesecure-sdk.git {SDK_ROOT}
else:
    !git -C {SDK_ROOT} pull

print('[OK] SDK:', SDK_ROOT)

## Step 4. CosyVoice 설치 (Drive 캐시)

- **처음 실행**: git clone + 모델 다운로드 → Drive 백업 (10~15분)
- **이후 실행**: Drive에서 복원 (2~3분)

In [ ]:
import os, shutil

COSYVOICE_ROOT  = '/content/CosyVoice'
COSYVOICE_DRIVE = '/content/drive/MyDrive/CosyVoice'

if os.path.exists(COSYVOICE_DRIVE):
    print('Drive 캐시에서 복원 중...')
    if os.path.exists(COSYVOICE_ROOT):
        shutil.rmtree(COSYVOICE_ROOT)
    shutil.copytree(COSYVOICE_DRIVE, COSYVOICE_ROOT)
    print('[OK] 복원 완료')
else:
    print('처음 설치 중 (git clone + 모델 다운로드)...')
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSYVOICE_ROOT}
    from huggingface_hub import snapshot_download
    snapshot_download('FunAudioLLM/Fun-CosyVoice3-0.5B-2512', local_dir=f'{COSYVOICE_ROOT}/pretrained_models/Fun-CosyVoice3-0.5B-2512')
    print('Drive에 백업 중...')
    if os.path.exists(COSYVOICE_DRIVE):
        shutil.rmtree(COSYVOICE_DRIVE)
    shutil.copytree(COSYVOICE_ROOT, COSYVOICE_DRIVE)
    print('[OK] Drive 백업 완료')

# torch/torchaudio/numpy/tensorrt는 Colab 기본값 유지, 나머지만 설치
!pip install -q conformer==0.3.2 deepspeed==0.15.1 diffusers==0.29.0 fastapi==0.115.6 fastapi-cli==0.0.4 gdown==5.1.0 gradio==5.4.0 grpcio==1.57.0 grpcio-tools==1.57.0 hydra-core==1.3.2 HyperPyYAML==1.2.3 inflect==7.3.1 librosa==0.10.2 lightning==2.2.4 matplotlib==3.7.5 modelscope==1.20.0 networkx==3.1 omegaconf==2.3.0 onnx==1.16.0 openai-whisper==20231117 protobuf==4.25 pyarrow==18.1.0 pydantic==2.7.0 pyworld==0.3.4 rich==13.7.1 soundfile==0.12.1 tensorboard==2.14.0 transformers==4.51.3 x-transformers==2.11.24 uvicorn==0.30.0 wetext==0.0.4 wget==3.2
!pip install -q --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ onnxruntime-gpu==1.18.0
print('[OK] 패키지 설치 완료')

## Step 5. CosyVoice 로드

In [ ]:
import sys, time

sys.path.insert(0, COSYVOICE_ROOT)
sys.path.insert(0, f'{COSYVOICE_ROOT}/third_party/Matcha-TTS')

from cosyvoice.cli.cosyvoice import CosyVoice3

t0 = time.time()
model = CosyVoice3(f'{COSYVOICE_ROOT}/pretrained_models/Fun-CosyVoice3-0.5B-2512')

if hasattr(model, 'model') and hasattr(model.model, 'llm'):
    model.model.llm = model.model.llm.float()
    print('LLM float32 변환 완료')

print(f'[OK] CosyVoice3 로드 완료 ({time.time()-t0:.1f}s)')
print(f'sample_rate: {model.sample_rate}')

## Step 6. 음성 로드 + 클로닝

셀 실행 후 입력창에:
- **레퍼런스 음성에서 실제로 하는 말**: 음성 파일 내용 그대로 (전사)
- **새로 합성할 문장**: 클로닝 목소리로 생성할 텍스트

In [ ]:
import numpy as np
import soundfile as sf
import tempfile, os, time
from math import gcd
from scipy.signal import resample_poly

SR = 16000

# 음성 로드 (16kHz mono float32)
data, src_sr = sf.read(AUDIO_PATH, dtype='float32', always_2d=True)
audio = data.mean(axis=1).astype(np.float32)
if src_sr != SR:
    g = gcd(SR, src_sr)
    audio = resample_poly(audio, SR // g, src_sr // g).astype(np.float32)
audio = np.clip(audio, -1.0, 1.0)
print(f'입력 음성: {len(audio)/SR:.2f}초')

# 사용자 입력
PROMPT_TEXT = input('레퍼런스 음성에서 실제로 하는 말 : ')
TTS_TEXT    = input('새로 합성할 문장 : ')

# 임시 파일에 저장 후 클로닝
with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
    sf.write(tmp.name, audio, SR)
    ref_path = tmp.name

end_tag = '<|endofprompt|>'
prompt_text = PROMPT_TEXT if PROMPT_TEXT.endswith(end_tag) else PROMPT_TEXT + end_tag

print('클로닝 중...')
t0 = time.time()
chunks = []
for result in model.inference_zero_shot(TTS_TEXT, prompt_text, ref_path, stream=False):
    chunks.append(result['tts_speech'].squeeze().numpy())
os.unlink(ref_path)

cloned = np.concatenate(chunks).astype(np.float32)
cloned = np.clip(cloned, -1.0, 1.0)
print(f'클로닝 완료 ({time.time()-t0:.1f}s) -> {len(cloned)/model.sample_rate:.2f}초')

## Step 7. 결과 저장 + 청취

In [ ]:
import soundfile as sf
from IPython.display import Audio, display
from pathlib import Path

stem = Path(AUDIO_PATH).stem
out_original = f'{OUTPUT_DIR}/{stem}_original.wav'
out_cloned   = f'{OUTPUT_DIR}/{stem}_cloned.wav'

sf.write(out_original, audio, SR)
sf.write(out_cloned,   cloned, model.sample_rate)
print('저장 완료:')
print(f'  원본 : {out_original}')
print(f'  클론 : {out_cloned}')

print('\n=== 원본 ===')
display(Audio(audio, rate=SR))
print('=== 클론 (CosyVoice3) ===')
display(Audio(cloned, rate=model.sample_rate))

## Step 8. 화자 임베딩 거리 확인

CosyVoice 내부 CAM++ 임베딩으로 원본 vs 클론 유사도를 측정합니다.

| cosine distance | 판정 |
|---|---|
| < 0.10 | 같은 화자 — 클로닝 성공 |
| 0.10 ~ 0.30 | 부분 방어 |
| > 0.30 | 다른 화자 — 방어 성공 |

In [ ]:
import numpy as np
import soundfile as sf
import tempfile, os

with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
    sf.write(tmp.name, audio, SR)
    orig_emb = model.frontend._extract_spk_embedding(tmp.name)
    os.unlink(tmp.name)

with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
    sf.write(tmp.name, cloned, model.sample_rate)
    clone_emb = model.frontend._extract_spk_embedding(tmp.name)
    os.unlink(tmp.name)

orig_np  = orig_emb.squeeze().cpu().numpy().astype(np.float32)
clone_np = clone_emb.squeeze().cpu().numpy().astype(np.float32)

cos_sim  = float(np.dot(orig_np, clone_np) / (np.linalg.norm(orig_np) * np.linalg.norm(clone_np) + 1e-8))
cos_dist = 1.0 - cos_sim

print(f'cosine similarity : {cos_sim:.4f}')
print(f'cosine distance   : {cos_dist:.4f}')
print()
if cos_dist < 0.10:
    print('결과: 같은 화자 — 클로닝 성공')
elif cos_dist > 0.30:
    print('결과: 다른 화자 — 방어 성공')
else:
    print('결과: 애매 (부분 방어)')